- [x] удалить строки с пустым title
- [x] разбить на леммы и морфемы
- [x] дисбаланс, будем использовать stratify 
- [x] утечек данных нет

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
import os
from razdel import tokenize
import pymorphy3
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, hinge_loss, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, ParameterGrid
from time import time

In [2]:
df = pd.read_csv("../datasets/porn_detection/train.csv")
df = df.dropna(subset=['title'])

In [3]:
df.head()

,ID,url,title,label
0,0,m.kp.md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0
1,1,www.kp.by,Эта песня стала известна многим телезрителям б...,0
2,2,fanserials.tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0
3,3,colorbox.spb.ru,Не Беси Меня Картинки,0
4,4,tula-sport.ru,В Новомосковске сыграют следж-хоккеисты алекси...,0


In [4]:
df[(df["label"] == 0) & (df["url"].str.contains("porn"))].shape

(0, 4)

#### Заметим, что некоторые комбинации букв содержатся только в сайтах 18+ (porn, porevo)
В будущем будем все сайты с таким называнием отмечать как сайты 18+ (повысим recall)

In [5]:
# preprocessing
def tokenize_df(df):

    def lemmatize_text(text):
        tokens = [token.text for token in tokenize(text)]
        lemmas = []
        for token in tokens:
            if token.isalpha():
                parsed = morph.parse(token)[0]  # берем первый вариант разбора
                lemmas.append(parsed.normal_form)
        return " ".join(lemmas)
    
    df['tokens'] = df['title'].apply(lambda x: " ".join([token.text for token in tokenize(x)])) 
    morph = pymorphy3.MorphAnalyzer()   # эта штука может убирать названия фильмов и тд
    df['lemmatized'] = df['tokens'].apply(lemmatize_text)

In [6]:
tokenize_df(df)

In [7]:
encoder = TfidfVectorizer(max_df=0.8,
                          min_df=2,
                          ngram_range=(1, 2))
X = df["lemmatized"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X_train = encoder.fit_transform(X_train)
X_test = encoder.transform(X_test)
print(np.unique(y_train))
print(X_test.shape)

[0 1]
(27062, 103451)


### Partial fit

In [8]:
n_samples = X_train.shape[0]

version = 2
log_file_name = f"log_reg_train_part_fit_{version}_loggin.log"

os.makedirs("plots_part_fit", exist_ok=True)

logger = logging.getLogger(f"sgd_part_training_{version}")
logger.setLevel(logging.INFO)
if not logger.handlers:
    file_handler = logging.FileHandler(log_file_name, mode="a", encoding="utf-8")
    file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(file_handler)


batch_size_list = [256] 
n_iter_no_change_list = [5, 10] 
tol_list = [1e-3, 1e-4]
max_epochs = 1000
param_grid = {
    "loss": ["hinge", "log_loss"], 
    "penalty" : ["l1", "l2"],
    "alpha": [1e-6],
    "learning_rate": ["optimal"],
    "eta0": [1e-3, 1e-4]
}
model_num = 0
for bs in batch_size_list:
    for n_iter_no_change in n_iter_no_change_list:
        for tol in tol_list:
             for params in ParameterGrid(param_grid):
                iters_no_change = 0
                loss_history_train, loss_history_test, f1_history, recall_history = [], [], [], []
                model = SGDClassifier(
                    **params,
                    random_state = 42,
                    class_weight = dict(zip(np.unique(y_train), y_train.shape[0] / (2 * np.bincount(y_train)))),
                    warm_start = True
                    )

                print(f"Model num: {model_num}")
                logger.info(
                    f"=== New model #{model_num} | batch_size={bs}, "
                    f"n_iter_no_change={n_iter_no_change}, tol={tol}, params={params} ==="
                )
                time_start = time()

                STOP = False
                for epoch in range(max_epochs):
                    if STOP:
                        break

                    for idx in range(0, n_samples, bs):
                        start, stop = idx, min(idx+bs, n_samples)
                        X_batch = X_train[start:stop]
                        y_batch = y_train[start:stop]
                        if idx == 0 and epoch == 0:
                            model.partial_fit(X_batch, y_batch, np.unique(y_train))
                        else:
                            model.partial_fit(X_batch, y_batch)

                        if idx%(bs*1000) == 0:
                            y_pred_test = model.predict(X_test)
                            if params["loss"] == "log_loss": 
                                cur_loss_train = log_loss(y_train, model.predict_proba(X_train))
                                cur_loss_test = log_loss(y_test, model.predict_proba(X_test))
                            else:
                                cur_loss_train = hinge_loss(y_train, model.decision_function(X_train))
                                cur_loss_test = hinge_loss(y_test, model.decision_function(X_test))
                            loss_history_train.append(cur_loss_train)
                            loss_history_test.append(cur_loss_test)
                            f1_history.append(f1_score(y_test, y_pred_test))
                            recall_history.append(recall_score(y_test, y_pred_test))
                            logger.info(
                                f"Model {model_num} | epoch={epoch} idx={idx} | "
                                f"f1={f1_history[-1]:.4f} recall={recall_history[-1]:.4f} loss_train={cur_loss_train:.4f} "
                                f"loss_test={cur_loss_test:.4f}"
                            )
                            # проверка на остановку
                            if iters_no_change >= n_iter_no_change:
                                STOP = True
                                logger.info(f"Model {model_num} | early stopping at epoch={epoch} idx={idx}")
                                break
                            else:
                                if len(loss_history_train) >= 2 and tol > (loss_history_train[-2] - loss_history_train[-1]):
                                    iters_no_change += 1
                                else: 
                                    iters_no_change = 0

                    epoch_elapsed = time() - time_start
                    print(f"Эпоха №{epoch}: {epoch_elapsed}")
                    logger.info(f"Model {model_num} | epoch={epoch} finished | elapsed={epoch_elapsed:.2f}s")

                total_elapsed = time() - time_start
                print(f"Time elapsed for model {model_num}: {total_elapsed}")
                logger.info(f"Model {model_num} | training finished | total_elapsed={total_elapsed:.2f}s")

                fig, ax = plt.subplots()
                ax.plot(loss_history_train, label="train")
                ax.plot(loss_history_test, label="test")
                ax.set_xlabel("checkpoint")
                ax.set_ylabel("loss")
                ax.set_title(f"Model {model_num} loss (train vs test)")
                ax.legend()
                fig.savefig(f"plots_part_fit/model_{model_num}_loss.png")
                plt.close(fig)

                fig, ax = plt.subplots()
                ax.plot(f1_history, label="f1")
                ax.plot(recall_history, label="recall")
                ax.set_xlabel("checkpoint")
                ax.set_ylabel("score")
                ax.set_title(f"Model {model_num} f1 / recall")
                ax.legend()
                fig.savefig(f"plots_part_fit/model_{model_num}_f1.png")
                plt.close(fig)

                model_num+=1

Model num: 0
Эпоха №0: 1.2113747596740723
Эпоха №1: 1.9599459171295166
Эпоха №2: 2.8487329483032227
Эпоха №3: 3.702133893966675
Эпоха №4: 4.659965515136719
Эпоха №5: 5.41973352432251
Эпоха №6: 6.412298679351807
Эпоха №7: 7.539352655410767
Эпоха №8: 8.634592771530151
Эпоха №9: 9.73393702507019
Эпоха №10: 10.847938299179077
Эпоха №11: 11.775163650512695
Time elapsed for model 0: 11.775680541992188
Model num: 1
Эпоха №0: 0.7875807285308838
Эпоха №1: 1.6616971492767334
Эпоха №2: 2.4401214122772217
Эпоха №3: 3.3901891708374023
Эпоха №4: 4.491543769836426
Эпоха №5: 5.353398084640503
Эпоха №6: 6.138543605804443
Эпоха №7: 6.8823628425598145
Эпоха №8: 7.7524402141571045
Эпоха №9: 7.808849096298218
Time elapsed for model 1: 7.808991193771362
Model num: 2
Эпоха №0: 0.8810999393463135
Эпоха №1: 1.7719945907592773
Эпоха №2: 2.566422700881958
Эпоха №3: 3.419600486755371
Эпоха №4: 4.327889442443848
Эпоха №5: 5.796323537826538
Эпоха №6: 7.470837831497192
Эпоха №7: 9.0726478099823
Time elapsed for mode

In [9]:
# BEST MODELS
#61 | batch_size=128, n_iter_no_change=5, tol=0.0001, params={'alpha': 1e-05, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'eta0': 0.0001, 'learning_rate': 'optimal', 'loss': 'hinge', 'penalty': 'l2', 'random_state': 42, 'warm_start': True} ===

#49 | batch_size=128, n_iter_no_change=5, tol=0.0001, params={'alpha': 1e-05, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'eta0': 0.001, 'learning_rate': 'optimal', 'loss': 'hinge', 'penalty': 'l2', 'random_state': 42, 'warm_start': True} ===

#27 | batch_size=128, n_iter_no_change=5, tol=0.001, params={'alpha': 1e-06, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'eta0': 0.001, 'learning_rate': 'optimal', 'loss': 'log_loss', 'penalty': 'l2', 'random_state': 42, 'warm_start': True} ===

#13 | batch_size=128, n_iter_no_change=5, tol=0.001, params={'alpha': 1e-05, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'eta0': 0.0001, 'learning_rate': 'optimal', 'loss': 'hinge', 'penalty': 'l2', 'random_state': 42, 'warm_start': True} ===

#1  | batch_size=128, n_iter_no_change=5, tol=0.001, params={'alpha': 1e-05, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'eta0': 0.001, 'learning_rate': 'optimal', 'loss': 'hinge', 'penalty': 'l2', 'random_state': 42, 'warm_start': True} ===

#75 | batch_size=128, n_iter_no_change=5, tol=0.0001, params={'alpha': 1e-06, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'eta0': 0.001, 'learning_rate': 'optimal', 'loss': 'log_loss', 'penalty': 'l2', 'random_state': 42, 'warm_start': True} ===


### Fit

In [10]:
# n_samples = X_train.shape[0]

# os.makedirs("plots_fit", exist_ok=True)

# logger = logging.getLogger("sgd_training")
# logger.setLevel(logging.INFO)
# if not logger.handlers:
#     file_handler = logging.FileHandler("log_reg_train_fit_loggin.log", mode="a", encoding="utf-8")
#     file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
#     logger.addHandler(file_handler)

# param_grid = {
#     "loss": ["hinge", "log_loss"], 
#     "penalty" : ["l1", "l2"],
#     "alpha": [1e-5, 1e-6],
#     "learning_rate": ["optimal", "constant", "adaptive"],
#     "eta0": [1e-3, 1e-4], 
#     "n_iter_no_change": [5, 10], 
#     "tol": [1e-3, 1e-4, 1e-5],
#     "max_iter": [1000, 2000],
#     "validation_fraction": [0.1, 0.2]
# }

# model_num = 0
# for params in ParameterGrid(param_grid):
#     time_start = time()
#     model = SGDClassifier(**params, random_state = 42,
#         class_weight = dict(zip(np.unique(y_train), y_train.shape[0] / (2 * np.bincount(y_train)))),
#         early_stopping = True)

#     print(f"Model num: {model_num}")
#     logger.info(
#         f"=== New model #{model_num} "
#         f"params={params} ==="
#     )

#     model.fit(X_train, y_train)

#     y_pred_test = model.predict(X_test)
#     if params["loss"] == "log_loss": 
#         cur_loss_train = log_loss(y_train, model.predict_proba(X_train))
#         cur_loss_test = log_loss(y_test, model.predict_proba(X_test))
#     else:
#         cur_loss_train = hinge_loss(y_train, model.decision_function(X_train))
#         cur_loss_test = hinge_loss(y_test, model.decision_function(X_test))
#     logger.info(
#         f"Model {model_num}| "
#         f"f1={f1_score(y_test, y_pred_test):.4f} recall={recall_score(y_test, y_pred_test):.4f} loss_train={cur_loss_train:.4f} "
#         f"loss_test={cur_loss_test:.4f}"
#     )

#     logger.info(f"Model {model_num} | training finished | total_elapsed={(time() - time_start):.2f}s")
#     model_num+=1


